# Music Annotation Project V8.1
## Full-Pixel, Whole-Page, Source-Group Out-of-Fold Evaluation — revised artefact discovery

This revision fixes the earlier artefact-resolution problem rather than changing the experiment. It searches V8.0b/c/d project roots broadly, supports `fold_map.json` as well as CSV, and distinguishes Stage-B OOF artefacts from the later Stage-C frozen bundle.

**Fail-closed rule:** the notebook never recreates folds, calibration, schema, ensemble/OOD objects or thresholds from outcomes. If an artefact scientifically required for inference cannot be recovered unambiguously, the run stops after writing the artefact census.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-learn joblib torch torchvision psutil


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, os, sys, platform, time, re, warnings
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, psutil
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_1'
V8_SEARCH_ROOTS=[ROOT/'observed_print_classifier_v8_0b',ROOT/'observed_print_classifier_v8_0c',ROOT/'observed_print_classifier_v8_0d']

# Optional exact overrides. Leave None for discovery.
V75_RUN_OVERRIDE=None
V8_RUN_OVERRIDE=None
FOLD_MAP_OVERRIDE=None
DEVELOPMENT_MANIFEST_OVERRIDE=None
SCHEMA_OVERRIDE=None
POLICY_OVERRIDE=None
REPLAY_OVERRIDE=None

MASK_DILATION=2; PATCH_SIZE=33; PATCH_RADIUS=16; OBSERVED_DARK_THRESHOLD=200
PRINT_THRESHOLD=0.995; ANNOTATION_THRESHOLD=0.995; PRINT_ANNOTATION_MARGIN=0.990; MIN_PRINT_RUN=2
BATCH_SIZE=4096; NUM_WORKERS=2; DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_1_full_pixel_oof_revised'
OUT=OUTPUT_ROOT/RUN
DIRS={x:OUT/x for x in ['01_input_audit','02_v78_equivalence','03_eligibility','04_oof_predictions','05_prediction_lock','06_truth_scoring','07_whole_pages','08_visual_audit','09_replay','failures']}
OUT.mkdir(parents=True,exist_ok=False)
for p in DIRS.values(): p.mkdir(parents=True,exist_ok=False)
print('Device:',DEVICE,'Output:',OUT)


## 2. Integrity helpers


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,v):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def fail(stage,message,details=None):
    rec={'created_utc':datetime.now(timezone.utc).isoformat(),'stage':stage,'message':message,'details':details}
    atomic_json(DIRS['failures']/f'{stage}_failure.json',rec); raise RuntimeError(f'{stage}: {message}')
def latest_complete(root,required):
    hits=[]; root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def image_map(folder):
    folder=Path(folder); return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}


## 3. Project-wide V8 artefact census


In [ ]:
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: fail('input_discovery','No complete V7.5 run found')
if V8_RUN_OVERRIDE is not None: V8_SEARCH_ROOTS.insert(0,Path(V8_RUN_OVERRIDE))
V8_SEARCH_ROOTS=[p for p in V8_SEARCH_ROOTS if p.exists()]
if not V8_SEARCH_ROOTS: fail('input_discovery','No V8.0b/c/d search roots exist')

patterns={
'inventory':['artifact_inventory.json','artefact_inventory.json','*artifact*inventory*.json','*artefact*inventory*.json'],
'fold_map':['fold_map.json','fold_map.csv','source_group_fold_map.json','source_group_fold_map.csv','*fold*map*.json','*fold*map*.csv'],
'development_manifest':['development_score_manifest.csv','development_score_manifest.json','paired_sample_manifest.csv','paired_sample_manifest.json','*development*manifest*.csv','*development*manifest*.json'],
'schema':['channel_schema.json','tensor_schema.json','predictor_schema.json','*channel*schema*.json','*tensor*schema*.json','*predictor*schema*.json'],
'policy':['policy.json','decision_policy.json','registered_operating_point_comparison.json','*policy*.json','*operating*point*.json'],
'experiment_manifest':['experiment_manifest.json','*manifest*.json'],
'split_audit':['source_group_split_audit.csv','source_group_split_audit.json','*split*audit*.csv','*split*audit*.json'],
'oof_predictions':['topology_oof_predictions.csv','control_oof_predictions.csv','*oof*prediction*.csv','*oof*prediction*.parquet'],
'checkpoint':['*.pt','*.pth','*.ckpt'],
'calibrator':['*calibrat*.joblib','*calibrat*.pkl','*calibrat*.pickle','*isotonic*.joblib','*isotonic*.pkl','*isotonic*.pickle','*calibrat*.json','*isotonic*.json'],
'ood':['*ood*.json','*ood*.joblib','*ood*.pkl','*distribution*.npz'],
'replay':['replay_test.json','*replay*.json','*replay*.npz']}

def dedupe(xs): return sorted(set(Path(x) for x in xs),key=lambda p:str(p))
found={k:[] for k in patterns}
for root in V8_SEARCH_ROOTS:
    for role,pats in patterns.items():
        for pat in pats: found[role].extend(p for p in root.rglob(pat) if p.is_file())
for k in found: found[k]=dedupe(found[k])
rows=[]
for role,paths in found.items():
    for p in paths: rows.append({'role':role,'name':p.name,'path':str(p),'parent':str(p.parent),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
census=pd.DataFrame(rows); census.to_csv(DIRS['01_input_audit']/'v8_project_wide_artifact_census.csv',index=False)
report={'search_roots':[str(p) for p in V8_SEARCH_ROOTS],'counts':{k:len(v) for k,v in found.items()},'candidates':{k:[str(p) for p in v] for k,v in found.items()}}
atomic_json(DIRS['01_input_audit']/'artifact_discovery.json',report)
print(json.dumps(report,indent=2))


## 4. Resolve fold map and development roster

This cell supports JSON or CSV fold maps. It will not generate new folds.


In [ ]:
def choose(override,paths,role,required=True):
    if override is not None:
        p=Path(override)
        if not p.exists(): fail('artifact_resolution',f'{role} override absent: {p}')
        return p
    paths=dedupe(paths)
    if len(paths)==1:return paths[0]
    if required: fail('artifact_resolution',f'Expected one unambiguous {role}; found {len(paths)}',[str(p) for p in paths])
    return None

def load_json_table(path,keys=('pages','rows','assignments','fold_map')):
    obj=json.loads(Path(path).read_text())
    if isinstance(obj,list): return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for key in keys:
            if isinstance(obj.get(key),list): return pd.DataFrame(obj[key])
        rows=[]
        for key,val in obj.items(): rows.append({'page':str(key),**val} if isinstance(val,dict) else {'page':str(key),'fold':val})
        return pd.DataFrame(rows)
    fail('json_table',f'Unsupported JSON root in {path}')

def load_table(path): return pd.read_csv(path) if Path(path).suffix.lower()=='.csv' else load_json_table(path)

FOLD_MAP=choose(FOLD_MAP_OVERRIDE,found['fold_map'],'fold map')
DEVELOPMENT_MANIFEST=choose(DEVELOPMENT_MANIFEST_OVERRIDE,found['development_manifest'],'development manifest')
folds=load_table(FOLD_MAP); manifest=load_table(DEVELOPMENT_MANIFEST)
aliases={'page_id':'page','page_name':'page','filename_stem':'page','group':'source_group','source':'source_group','score_group':'source_group','fold_id':'fold','outer_fold':'fold','test_fold':'fold'}
folds=folds.rename(columns={c:aliases[c] for c in folds.columns if c in aliases})
manifest=manifest.rename(columns={c:aliases[c] for c in manifest.columns if c in aliases})
if not {'page','fold'}.issubset(folds.columns): fail('fold_map','Fold map needs page and fold',list(folds.columns))
if not {'page','source_group'}.issubset(manifest.columns): fail('development_manifest','Manifest needs page and source_group',list(manifest.columns))
folds['page']=folds.page.astype(str); manifest['page']=manifest.page.astype(str)
if 'source_group' not in folds: folds=folds.merge(manifest[['page','source_group']],on='page',how='left',validate='one_to_one')
if folds.page.duplicated().any() or folds.source_group.isna().any(): fail('fold_map','Duplicate pages or unresolved source_group')
bad=folds.groupby('source_group').fold.nunique(); bad=bad[bad>1]
if len(bad): fail('fold_map','Source group split across folds',bad.to_dict())
folds.to_csv(DIRS['01_input_audit']/'resolved_fold_map.csv',index=False)
print('Fold map:',FOLD_MAP); print('Manifest:',DEVELOPMENT_MANIFEST); print('Rows:',len(folds),'groups:',folds.source_group.nunique(),'folds:',folds.fold.nunique())


## 5. Resolve the minimum OOF execution artefacts


In [ ]:
CHECKPOINTS=found['checkpoint']; CALIBRATORS=found['calibrator']; OOD_PATHS=found['ood']
SCHEMA_PATH=choose(SCHEMA_OVERRIDE,found['schema'],'channel schema',required=False)
POLICY_PATH=choose(POLICY_OVERRIDE,found['policy'],'decision policy',required=False)
REPLAY_PATH=choose(REPLAY_OVERRIDE,[p for p in found['replay'] if p.suffix.lower()=='.json'],'replay record',required=False)
if not CHECKPOINTS: fail('oof_bundle','No checkpoints found')
resolved=[]
for role,paths in found.items():
    for p in paths: resolved.append({'artifact_id':p.name,'role':role,'resolved_path':str(p),'sha256':sha256_file(p),'bytes':p.stat().st_size})
atomic_json(DIRS['01_input_audit']/'artifact_inventory_resolved.json',resolved)
print('Checkpoints:',len(CHECKPOINTS)); print('Calibrators:',len(CALIBRATORS)); print('Schema:',SCHEMA_PATH); print('Policy:',POLICY_PATH); print('Replay:',REPLAY_PATH)

# Stage-B may predate a packaged Stage-C policy/inventory, but inference still requires exact schema and calibration.
missing=[]
if SCHEMA_PATH is None: missing.append('schema')
if not CALIBRATORS: missing.append('calibrator')
if missing:
    fail('oof_bundle','Minimum OOF execution bundle is incomplete. Do not refit missing objects.',{'missing':missing,'census':str(DIRS['01_input_audit']/'v8_project_wide_artifact_census.csv')})

schema=json.loads(SCHEMA_PATH.read_text())
policy=json.loads(POLICY_PATH.read_text()) if POLICY_PATH else {'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'print_annotation_margin':PRINT_ANNOTATION_MARGIN,'minimum_print_run':MIN_PRINT_RUN,'_source':'registered constants; no separate policy artefact found'}


## 6. Stop point for the revised discovery build

If this cell is reached, the fold map, roster, checkpoints, schema and calibrator have all been found. The remaining original V8.1 scientific cells can then run against these resolved variables. This diagnostic build deliberately stops here if any minimum OOF artefact is missing instead of silently recreating it.


In [ ]:
summary={'v75_run':str(V75),'fold_map':str(FOLD_MAP),'development_manifest':str(DEVELOPMENT_MANIFEST),'checkpoint_count':len(CHECKPOINTS),'calibrator_count':len(CALIBRATORS),'schema':str(SCHEMA_PATH),'policy':str(POLICY_PATH) if POLICY_PATH else None,'replay':str(REPLAY_PATH) if REPLAY_PATH else None,'fold_rows':len(folds),'source_groups':int(folds.source_group.nunique()),'folds':int(folds.fold.nunique()),'status':'MINIMUM_OOF_BUNDLE_RESOLVED'}
atomic_json(OUT/'v8_1_revised_discovery_summary.json',summary)
print(json.dumps(summary,indent=2))
